# 02 · Silver transactions
Converts bronze transactions to real types, checks conversion quality, and adds stable IDs.

In [0]:
from pyspark.sql import functions as F

BRONZE = "workspace.aml_bronze"
SILVER = "workspace.aml_silver"

bronze = spark.table(f"{BRONZE}.transactions")

In [0]:
typed = bronze.select(
    F.expr("try_to_timestamp(timestamp, 'yyyy/MM/dd HH:mm')").alias("txn_timestamp"),
    F.trim("from_bank").alias("from_bank"),
    F.trim("from_account").alias("from_account"),
    F.trim("to_bank").alias("to_bank"),
    F.trim("to_account").alias("to_account"),
    F.expr("try_cast(amount_received AS DECIMAL(24,8))").alias("amount_received"),
    F.trim("receiving_currency").alias("receiving_currency"),
    F.expr("try_cast(amount_paid AS DECIMAL(24,8))").alias("amount_paid"),
    F.trim("payment_currency").alias("payment_currency"),
    F.trim("payment_format").alias("payment_format"),
    F.expr("try_cast(is_laundering AS INT)").alias("is_laundering"),
    "_source_file",
)

typed.printSchema()

In [0]:
checks = typed.agg(
    F.count("*").alias("rows"),
    F.sum(F.col("txn_timestamp").isNull().cast("int")).alias("bad_timestamp"),
    F.sum(F.col("amount_received").isNull().cast("int")).alias("bad_amount_received"),
    F.sum(F.col("amount_paid").isNull().cast("int")).alias("bad_amount_paid"),
    F.sum(F.col("is_laundering").isNull().cast("int")).alias("bad_label"),
)

display(checks)

In [0]:
display(
    typed.groupBy("is_laundering")
    .count()
    .orderBy("is_laundering")
)